In [1]:
import os
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.utils import shuffle

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb
from xgboost import XGBClassifier # Old version of xgboost: Version 1.7.6
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

## **Ensemble Learning**

In [ ]:
_df_train = pd.read_csv("train-metadata.csv")
_df_test = pd.read_csv("test/test-metadata.csv")

In [3]:
_df_test.shape

(3, 44)

In [4]:
train_path = 'train-metadata.csv'
test_path = '/test/test-metadata.csv'

id_col = 'isic_id'
target_col = 'target'
group_col = 'patient_id'

err = 1e-5
sampling_ratio = 0.01
seed = 42

In [5]:
num_cols = [
    'age_approx',                        # Approximate age of patient at time of imaging.
    'clin_size_long_diam_mm',            # Maximum diameter of the lesion (mm).+
    'tbp_lv_A',                          # A inside  lesion.+
    'tbp_lv_Aext',                       # A outside lesion.+
    'tbp_lv_B',                          # B inside  lesion.+
    'tbp_lv_Bext',                       # B outside lesion.+ 
    'tbp_lv_C',                          # Chroma inside  lesion.+
    'tbp_lv_Cext',                       # Chroma outside lesion.+
    'tbp_lv_H',                          # Hue inside the lesion; calculated as the angle of A* and B* in LAB* color space. Typical values range from 25 (red) to 75 (brown).+
    'tbp_lv_Hext',                       # Hue outside lesion.+
    'tbp_lv_L',                          # L inside lesion.+
    'tbp_lv_Lext',                       # L outside lesion.+
    'tbp_lv_areaMM2',                    # Area of lesion (mm^2).+
    'tbp_lv_area_perim_ratio',           # Border jaggedness, the ratio between lesions perimeter and area. Circular lesions will have low values; irregular shaped lesions will have higher values. Values range 0-10.+
    'tbp_lv_color_std_mean',             # Color irregularity, calculated as the variance of colors within the lesion's boundary.
    'tbp_lv_deltaA',                     # Average A contrast (inside vs. outside lesion).+
    'tbp_lv_deltaB',                     # Average B contrast (inside vs. outside lesion).+
    'tbp_lv_deltaL',                     # Average L contrast (inside vs. outside lesion).+
    'tbp_lv_deltaLB',                    #
    'tbp_lv_deltaLBnorm',                # Contrast between the lesion and its immediate surrounding skin. Low contrast lesions tend to be faintly visible such as freckles; high contrast lesions tend to be those with darker pigment. Calculated as the average delta LB of the lesion relative to its immediate background in LAB* color space. Typical values range from 5.5 to 25.+
    'tbp_lv_eccentricity',               # Eccentricity.+
    'tbp_lv_minorAxisMM',                # Smallest lesion diameter (mm).+
    'tbp_lv_nevi_confidence',            # Nevus confidence score (0-100 scale) is a convolutional neural network classifier estimated probability that the lesion is a nevus. The neural network was trained on approximately 57,000 lesions that were classified and labeled by a dermatologist.+,++
    'tbp_lv_norm_border',                # Border irregularity (0-10 scale); the normalized average of border jaggedness and asymmetry.+
    'tbp_lv_norm_color',                 # Color variation (0-10 scale); the normalized average of color asymmetry and color irregularity.+
    'tbp_lv_perimeterMM',                # Perimeter of lesion (mm).+
    'tbp_lv_radial_color_std_max',       # Color asymmetry, a measure of asymmetry of the spatial distribution of color within the lesion. This score is calculated by looking at the average standard deviation in LAB* color space within concentric rings originating from the lesion center. Values range 0-10.+
    'tbp_lv_stdL',                       # Standard deviation of L inside  lesion.+
    'tbp_lv_stdLExt',                    # Standard deviation of L outside lesion.+
    'tbp_lv_symm_2axis',                 # Border asymmetry; a measure of asymmetry of the lesion's contour about an axis perpendicular to the lesion's most symmetric axis. Lesions with two axes of symmetry will therefore have low scores (more symmetric), while lesions with only one or zero axes of symmetry will have higher scores (less symmetric). This score is calculated by comparing opposite halves of the lesion contour over many degrees of rotation. The angle where the halves are most similar identifies the principal axis of symmetry, while the second axis of symmetry is perpendicular to the principal axis. Border asymmetry is reported as the asymmetry value about this second axis. Values range 0-10.+
    'tbp_lv_symm_2axis_angle',           # Lesion border asymmetry angle.+
    'tbp_lv_x',                          # X-coordinate of the lesion on 3D TBP.+
    'tbp_lv_y',                          # Y-coordinate of the lesion on 3D TBP.+
    'tbp_lv_z',                          # Z-coordinate of the lesion on 3D TBP.+
]

In [6]:

new_num_cols = [
    'lesion_size_ratio',             # tbp_lv_minorAxisMM      / clin_size_long_diam_mm
    'lesion_shape_index',            # tbp_lv_areaMM2          / tbp_lv_perimeterMM **2
    'hue_contrast',                  # tbp_lv_H                - tbp_lv_Hext              abs
    'luminance_contrast',            # tbp_lv_L                - tbp_lv_Lext              abs
    'lesion_color_difference',       # tbp_lv_deltaA **2       + tbp_lv_deltaB **2 + tbp_lv_deltaL **2  sqrt  
    'border_complexity',             # tbp_lv_norm_border      + tbp_lv_symm_2axis
    'color_uniformity',              # tbp_lv_color_std_mean   / tbp_lv_radial_color_std_max

    'position_distance_3d',          # tbp_lv_x **2 + tbp_lv_y **2 + tbp_lv_z **2  sqrt
    'perimeter_to_area_ratio',       # tbp_lv_perimeterMM      / tbp_lv_areaMM2
    'area_to_perimeter_ratio',       # tbp_lv_areaMM2          / tbp_lv_perimeterMM
    'lesion_visibility_score',       # tbp_lv_deltaLBnorm      + tbp_lv_norm_color
    'symmetry_border_consistency',   # tbp_lv_symm_2axis       * tbp_lv_norm_border
    'consistency_symmetry_border',   # tbp_lv_symm_2axis       * tbp_lv_norm_border / (tbp_lv_symm_2axis + tbp_lv_norm_border)

    'color_consistency',             # tbp_lv_stdL             / tbp_lv_Lext
    'consistency_color',             # tbp_lv_stdL*tbp_lv_Lext / tbp_lv_stdL + tbp_lv_Lext
    'size_age_interaction',          # clin_size_long_diam_mm  * age_approx
    'hue_color_std_interaction',     # tbp_lv_H                * tbp_lv_color_std_mean
    'lesion_severity_index',         # tbp_lv_norm_border      + tbp_lv_norm_color + tbp_lv_eccentricity / 3
    'shape_complexity_index',        # border_complexity       + lesion_shape_index
    'color_contrast_index',          # tbp_lv_deltaA + tbp_lv_deltaB + tbp_lv_deltaL + tbp_lv_deltaLBnorm

    'log_lesion_area',               # tbp_lv_areaMM2          + 1  np.log
    'normalized_lesion_size',        # clin_size_long_diam_mm  / age_approx
    'mean_hue_difference',           # tbp_lv_H                + tbp_lv_Hext    / 2
    'std_dev_contrast',              # tbp_lv_deltaA **2 + tbp_lv_deltaB **2 + tbp_lv_deltaL **2   / 3  np.sqrt
    'color_shape_composite_index',   # tbp_lv_color_std_mean   + bp_lv_area_perim_ratio + tbp_lv_symm_2axis   / 3
    'lesion_orientation_3d',         # tbp_lv_y                , tbp_lv_x  np.arctan2
    'overall_color_difference',      # tbp_lv_deltaA           + tbp_lv_deltaB + tbp_lv_deltaL   / 3

    'symmetry_perimeter_interaction',# tbp_lv_symm_2axis       * tbp_lv_perimeterMM
    'comprehensive_lesion_index',    # tbp_lv_area_perim_ratio + tbp_lv_eccentricity + bp_lv_norm_color + tbp_lv_symm_2axis   / 4
    'color_variance_ratio',          # tbp_lv_color_std_mean   / tbp_lv_stdLExt
    'border_color_interaction',      # tbp_lv_norm_border      * tbp_lv_norm_color
    'border_color_interaction_2',
    'size_color_contrast_ratio',     # clin_size_long_diam_mm  / tbp_lv_deltaLBnorm
    'age_normalized_nevi_confidence',# tbp_lv_nevi_confidence  / age_approx
    'age_normalized_nevi_confidence_2',
    'color_asymmetry_index',         # tbp_lv_symm_2axis       * tbp_lv_radial_color_std_max

    'volume_approximation_3d',       # tbp_lv_areaMM2          * sqrt(tbp_lv_x**2 + tbp_lv_y**2 + tbp_lv_z**2)
    'color_range',                   # abs(tbp_lv_L - tbp_lv_Lext) + abs(tbp_lv_A - tbp_lv_Aext) + abs(tbp_lv_B - tbp_lv_Bext)
    'shape_color_consistency',       # tbp_lv_eccentricity     * tbp_lv_color_std_mean
    'border_length_ratio',           # tbp_lv_perimeterMM      / pi * sqrt(tbp_lv_areaMM2 / pi)
    'age_size_symmetry_index',       # age_approx              * clin_size_long_diam_mm * tbp_lv_symm_2axis
    'index_age_size_symmetry',       # age_approx              * tbp_lv_areaMM2 * tbp_lv_symm_2axis
]

In [7]:
cat_cols = ['sex', 'anatom_site_general', 'tbp_tile_type', 'tbp_lv_location', 'tbp_lv_location_simple', 'attribution']
norm_cols = [f'{col}_patient_norm' for col in num_cols + new_num_cols]
special_cols = ['count_per_patient'] + [f'{col}_count' for col in cat_cols] + [(f'{num_col}_{cat_col}') for num_col, cat_col in itertools.product(num_cols, cat_cols)]
feature_cols = num_cols + new_num_cols + cat_cols + norm_cols + special_cols

In [8]:
def load_data(df):
    return (
        df
        .with_columns(
            pl.col('age_approx').cast(pl.String).replace('NA', np.nan).cast(pl.Float64),
        )

        # Fill all NaNs in float columns with their median
        .with_columns(
            pl.col(pl.Float64).fill_nan(pl.col(pl.Float64).median()),
        )

        # Fill all NaNs in categorical columns with their most frequent value
        .with_columns(
            pl.col(pl.Categorical).fill_nan(pl.col(pl.Categorical).mode()),
        )
        
        # First batch of feature engineering
        .with_columns(
            lesion_size_ratio           = pl.col('tbp_lv_minorAxisMM') / pl.col('clin_size_long_diam_mm'),
            lesion_shape_index          = pl.col('tbp_lv_areaMM2') / (pl.col('tbp_lv_perimeterMM') ** 2),
            hue_contrast                = (pl.col('tbp_lv_H') - pl.col('tbp_lv_Hext')).abs(),
            luminance_contrast          = (pl.col('tbp_lv_L') - pl.col('tbp_lv_Lext')).abs(),
            lesion_color_difference     = (pl.col('tbp_lv_deltaA') ** 2 + pl.col('tbp_lv_deltaB') ** 2 + pl.col('tbp_lv_deltaL') ** 2).sqrt(),
            border_complexity           = pl.col('tbp_lv_norm_border') + pl.col('tbp_lv_symm_2axis'),
            color_uniformity            = pl.col('tbp_lv_color_std_mean') / (pl.col('tbp_lv_radial_color_std_max') + 1e-8),
        )

        # Second batch of derived geometrical/color features
        .with_columns(
            position_distance_3d        = (pl.col('tbp_lv_x') ** 2 + pl.col('tbp_lv_y') ** 2 + pl.col('tbp_lv_z') ** 2).sqrt(),
            perimeter_to_area_ratio     = pl.col('tbp_lv_perimeterMM') / pl.col('tbp_lv_areaMM2'),
            area_to_perimeter_ratio     = pl.col('tbp_lv_areaMM2') / pl.col('tbp_lv_perimeterMM'),
            lesion_visibility_score     = pl.col('tbp_lv_deltaLBnorm') + pl.col('tbp_lv_norm_color'),
            combined_anatomical_site    = pl.col('anatom_site_general') + '_' + pl.col('tbp_lv_location'),
            symmetry_border_consistency = pl.col('tbp_lv_symm_2axis') * pl.col('tbp_lv_norm_border'),
            consistency_symmetry_border = pl.col('tbp_lv_symm_2axis') * pl.col('tbp_lv_norm_border') / (pl.col('tbp_lv_symm_2axis') + pl.col('tbp_lv_norm_border') + 1e-8),
        )

        # More color and interaction indices
        .with_columns(
            color_consistency           = pl.col('tbp_lv_stdL') / pl.col('tbp_lv_Lext'),
            consistency_color           = pl.col('tbp_lv_stdL') * pl.col('tbp_lv_Lext') / (pl.col('tbp_lv_stdL') + pl.col('tbp_lv_Lext') + 1e-8),
            size_age_interaction        = pl.col('clin_size_long_diam_mm') * pl.col('age_approx'),
            hue_color_std_interaction   = pl.col('tbp_lv_H') * pl.col('tbp_lv_color_std_mean'),
            lesion_severity_index       = (pl.col('tbp_lv_norm_border') + pl.col('tbp_lv_norm_color') + pl.col('tbp_lv_eccentricity')) / 3,
            shape_complexity_index      = pl.col('border_complexity') + pl.col('lesion_shape_index'),
            color_contrast_index        = pl.col('tbp_lv_deltaA') + pl.col('tbp_lv_deltaB') + pl.col('tbp_lv_deltaL') + pl.col('tbp_lv_deltaLBnorm'),
        )

        # Log and mean derived measurements
        .with_columns(
            log_lesion_area             = (pl.col('tbp_lv_areaMM2') + 1).log(),
            normalized_lesion_size      = pl.col('clin_size_long_diam_mm') / pl.col('age_approx'),
            mean_hue_difference         = (pl.col('tbp_lv_H') + pl.col('tbp_lv_Hext')) / 2,
            std_dev_contrast            = ((pl.col('tbp_lv_deltaA') ** 2 + pl.col('tbp_lv_deltaB') ** 2 + pl.col('tbp_lv_deltaL') ** 2) / 3).sqrt(),
            color_shape_composite_index = (pl.col('tbp_lv_color_std_mean') + pl.col('tbp_lv_area_perim_ratio') + pl.col('tbp_lv_symm_2axis')) / 3,
            lesion_orientation_3d       = pl.arctan2(pl.col('tbp_lv_y'), pl.col('tbp_lv_x')),
            overall_color_difference    = (pl.col('tbp_lv_deltaA') + pl.col('tbp_lv_deltaB') + pl.col('tbp_lv_deltaL')) / 3,
        )

        # Final group of compound indices
        .with_columns(
            symmetry_perimeter_interaction = pl.col('tbp_lv_symm_2axis') * pl.col('tbp_lv_perimeterMM'),
            comprehensive_lesion_index     = (pl.col('tbp_lv_area_perim_ratio') + pl.col('tbp_lv_eccentricity') + pl.col('tbp_lv_norm_color') + pl.col('tbp_lv_symm_2axis')) / 4,
            color_variance_ratio           = pl.col('tbp_lv_color_std_mean') / pl.col('tbp_lv_stdLExt'),
            border_color_interaction       = pl.col('tbp_lv_norm_border') * pl.col('tbp_lv_norm_color'),
            border_color_interaction_2     = pl.col('tbp_lv_norm_border') * pl.col('tbp_lv_norm_color') / (pl.col('tbp_lv_norm_border') + pl.col('tbp_lv_norm_color') + 1e-8),
            size_color_contrast_ratio      = pl.col('clin_size_long_diam_mm') / pl.col('tbp_lv_deltaLBnorm'),
            age_normalized_nevi_confidence = pl.col('tbp_lv_nevi_confidence') / pl.col('age_approx'),
            age_normalized_nevi_confidence_2 = (pl.col('clin_size_long_diam_mm')**2 + pl.col('age_approx')**2).sqrt(),
            color_asymmetry_index          = pl.col('tbp_lv_radial_color_std_max') * pl.col('tbp_lv_symm_2axis'),
        )

        # Final geometry and shape features
        .with_columns(
            volume_approximation_3d     = pl.col('tbp_lv_areaMM2') * (pl.col('tbp_lv_x')**2 + pl.col('tbp_lv_y')**2 + pl.col('tbp_lv_z')**2).sqrt(),
            color_range                 = (pl.col('tbp_lv_L') - pl.col('tbp_lv_Lext')).abs() + (pl.col('tbp_lv_A') - pl.col('tbp_lv_Aext')).abs() + (pl.col('tbp_lv_B') - pl.col('tbp_lv_Bext')).abs(),
            shape_color_consistency     = pl.col('tbp_lv_eccentricity') * pl.col('tbp_lv_color_std_mean'),
            border_length_ratio         = pl.col('tbp_lv_perimeterMM') / (2 * np.pi * (pl.col('tbp_lv_areaMM2') / np.pi).sqrt()),
            age_size_symmetry_index     = pl.col('age_approx') * pl.col('clin_size_long_diam_mm') * pl.col('tbp_lv_symm_2axis'),
            index_age_size_symmetry     = pl.col('age_approx') * pl.col('tbp_lv_areaMM2') * pl.col('tbp_lv_symm_2axis'),
        )

        # Patient-level z-score normalization
        .with_columns([
            ((pl.col(col) - pl.col(col).mean().over('patient_id')) / (pl.col(col).std().over('patient_id') + 1e-8)).alias(f'{col}_patient_norm')
            for col in (num_cols + new_num_cols)
        ])

        # Patient + category normalization
        .with_columns([
            ((pl.col(num_col) - pl.col(num_col).mean().over('patient_id', cat_col)) /
             (pl.col(num_col).std().over('patient_id', cat_col) + 1e-8)).alias(f'{num_col}_{cat_col}')
            for num_col, cat_col in itertools.product(num_cols, cat_cols)
        ])

        # Count features per group
        .with_columns([
            pl.col(col).count().over('patient_id', col).alias(f'{col}_count') for col in cat_cols
        ])

        # Count total samples per patient
        .with_columns(
            count_per_patient = pl.col('isic_id').count().over('patient_id'),
        )

        # Cast all categorical columns
        .with_columns(
            pl.col(cat_cols).cast(pl.Categorical),
        )

        # Convert to pandas and set index
        .to_pandas()
        .set_index(id_col)
    )


In [9]:
def preprocess(df_train, df_test):
    """
    Preprocess the data by applying one-hot encoding to categorical features.
    """
    global cat_cols
    
    encoder = OneHotEncoder(sparse_output=False, dtype=np.int32, handle_unknown='ignore')
    encoder.fit(df_train[cat_cols])
    
    new_cat_cols = [f'onehot_{i}' for i in range(len(encoder.get_feature_names_out()))]

    df_train[new_cat_cols] = encoder.transform(df_train[cat_cols])
    df_train[new_cat_cols] = df_train[new_cat_cols].astype('category')

    df_test[new_cat_cols] = encoder.transform(df_test[cat_cols])
    df_test[new_cat_cols] = df_test[new_cat_cols].astype('category')

    for col in cat_cols:
        feature_cols.remove(col)

    feature_cols.extend(new_cat_cols)
    cat_cols = new_cat_cols
    
    return df_train, df_test

In [10]:
def custom_metric(estimator, X, y_true):
    """
    Custom metric function to calculate the partial AUC.
    """
    y_hat = estimator.predict_proba(X)[:, 1]
    min_tpr = 0.80
    max_fpr = abs(1 - min_tpr)
    
    v_gt = abs(y_true - 1)
    v_pred = np.array([1.0 - x for x in y_hat])
    
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    
    return partial_auc

### **Loading Data & Feature Engineering**

In [11]:
df_train = load_data(pl.from_pandas(_df_train))
df_test = load_data(pl.from_pandas(_df_test))

In [12]:
df_train, df_test = preprocess(df_train, df_test)

In [13]:
# check which columns have NaN values
len(df_train.columns[df_train.isna().any()].tolist())

297

In [14]:
# show the shapoe of the data
df_train.shape, df_test.shape

((401059, 431), (3, 420))

In [15]:
# Deduplicate exact rows (based on all columns except the target)
df_train_dedup = df_train.drop_duplicates()

# Now split with no risk of duplicates leaking across sets
df_train_final, df_temp = train_test_split(
    df_train_dedup, test_size=0.30, random_state=42, stratify=df_train_dedup[target_col]
)

df_val, df_test_final = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp[target_col]
)

In [16]:
df_train_final.shape, df_val.shape, df_test_final.shape

((280741, 431), (60159, 431), (60159, 431))

In [17]:
df_train_final['target'].value_counts()

target
0    280466
1       275
Name: count, dtype: int64

In [18]:
df_test_final['target'].value_counts()

target
0    60100
1       59
Name: count, dtype: int64

In [19]:
df_val['target'].value_counts()

target
0    60100
1       59
Name: count, dtype: int64

### **HyperParam Tuned Models**

In [20]:
sampling_ratio = 0.01
seed = 42

# Parameters for the light gradient boosting model
lgb_params = {
    'objective':        'binary',
    'verbosity':        -1,
    'n_iter':           200,
    'n_jobs':           2,
    'boosting_type':    'gbdt',
    'lambda_l1':        0.03335206514282942, 
    'lambda_l2':        0.005157393323802471, 
    'learning_rate':    0.030665870185795318, 
    'max_depth':        7, 
    'num_leaves':       239, 
    'colsample_bytree': 0.7573175155547233, 
    'colsample_bynode': 0.5005423904042993, 
    'bagging_fraction': 0.7937347683420382, 
    'bagging_freq':     4, 
    'min_data_in_leaf': 29, 
    'scale_pos_weight': 1.648349898918236,
}

# Parameters for the CatBoost model
cat_params = {
    'loss_function':     'Logloss',
    'iterations':        250,
    'verbose':           False,
    'random_state':      32,
    'max_depth':         7, 
    'learning_rate':     0.06936242010150652, 
    'scale_pos_weight':  2.6149345838209532, 
    'l2_leaf_reg':       6.216113851699493, 
    'subsample':         0.6249261779711819, 
    'min_data_in_leaf':  24,
    'cat_features':      cat_cols,
}

# Parameters for the XGBoost model
xgb_params = {
    'enable_categorical': True,
    'tree_method':        'hist',
    'random_state':       42,
    'learning_rate':      0.08501257473292347, 
    'lambda':             8.879624125465703, 
    'alpha':              0.6779926606782505, 
    'max_depth':          6, 
    'subsample':          0.6012681388711075, 
    'colsample_bytree':   0.8437772277074493, 
    'colsample_bylevel':  0.5476090898823716, 
    'colsample_bynode':   0.9928601203635129, 
    'scale_pos_weight':   3.29440313334688,
}

In [21]:
# ========== Preprocessing for LGBM and XGBoost ==========
categorical_transformer_common = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_common = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', categorical_transformer_common, cat_cols)
])

# ========== Custom DataFrame Wrapper for CatBoost ==========
class DataFrameWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.columns)

# ========== Preprocessing for CatBoost (No OneHotEncoding) ==========
catboost_transformer = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols)
])

preprocessor_catboost = SklearnPipeline([
    ('transformer', catboost_transformer),
    ('to_df', DataFrameWrapper(columns=num_cols + cat_cols))
])

In [22]:
lgb_model = Pipeline([
    ('sampler_1', RandomOverSampler(sampling_strategy= 0.003 , random_state=22)),
    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=22)),
    ('classifier', lgb.LGBMClassifier(**lgb_params)),
])

cat_model = Pipeline([
    ('sampler_1', RandomOverSampler(sampling_strategy= 0.003 , random_state=32)),
    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=32)),
    ('classifier', CatBoostClassifier(**cat_params)),
])

xgb_model = Pipeline([
    ('sampler_1', RandomOverSampler(sampling_strategy= 0.003 , random_state=seed)),
    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=seed)),
    ('classifier', xgb.XGBClassifier(**xgb_params))
])

# ========== LightGBM pipeline with SMOTE==========
lgb_model_smote = Pipeline([
    ('preprocessor', preprocessor_common),
    ('sampler_1', SMOTE(sampling_strategy=0.003, random_state=22)),
    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=22)),
    ('classifier', lgb.LGBMClassifier(**lgb_params))
])

# ========== XGBoost pipeline WITH SMOTE ==========
xgb_model_smote = Pipeline([
    ('preprocessor', preprocessor_common),
    ('sampler_1', SMOTE(sampling_strategy=0.003, random_state=seed)),
    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=seed)),
    ('classifier', xgb.XGBClassifier(**xgb_params))
])

# ========== CatBoost pipeline with SMOTE ========== SMOTE DOESN'T WORK WITH CATBOOST
# cat_model_smote = Pipeline([
#    ('preprocessor', preprocessor_catboost),
#    ('sampler_1', SMOTE(sampling_strategy=0.003, random_state=32)),
#    ('sampler_2', RandomUnderSampler(sampling_strategy=sampling_ratio, random_state=32)),
#    ('classifier', CatBoostClassifier(**cat_params))
#])

In [23]:
estimator = VotingClassifier([
    ('lgb', lgb_model_smote),
    ('cat', cat_model),
    ('xgb', xgb_model_smote)
], voting='soft')

### **Cross-validation**

In [24]:
X = df_train_final[feature_cols]
y = df_train_final[target_col]
groups = df_train_final[group_col]
cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)

In [25]:
val_score = cross_val_score(
    estimator=estimator, 
    X=X, y=y, 
    cv=cv, 
    groups=groups,
    scoring=custom_metric,
)

np.mean(val_score), val_score

(0.17192011336777532,
 array([0.17395864, 0.17716258, 0.18195657, 0.17515734, 0.15136544]))

In [26]:
X, y = df_train_final[feature_cols], df_train_final[target_col]

estimator.fit(X, y)

VotingClassifier(estimators=[('lgb',
                              Pipeline(steps=[('preprocessor',
                                               ColumnTransformer(transformers=[('num',
                                                                                SimpleImputer(strategy='median'),
                                                                                ['age_approx',
                                                                                 'clin_size_long_diam_mm',
                                                                                 'tbp_lv_A',
                                                                                 'tbp_lv_Aext',
                                                                                 'tbp_lv_B',
                                                                                 'tbp_lv_Bext',
                                                                                 'tbp_lv_C',
                                                                                 'tbp_lv_Cext',
                                                                                 'tbp_lv_H',
                                                                                 'tbp_lv_Hext',
                                                                                 'tbp_lv_L',
                                                                                 'tbp_lv_Lext',
                                                                                 'tbp_lv_areaMM2',
                                                                                 'tbp_lv_area_perim_ratio',
                                                                                 'tb...
                                                             grow_policy=None,
                                                             importance_type=None,
                                                             interaction_constraints=None,
                                                             lambda=8.879624125465703,
                                                             learning_rate=0.08501257473292347,
                                                             max_bin=None,
                                                             max_cat_threshold=None,
                                                             max_cat_to_onehot=None,
                                                             max_delta_step=None,
                                                             max_depth=6,
                                                             max_leaves=None,
                                                             min_child_weight=None,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=None, ...))]))],
                 voting='soft')

In [27]:
estimator.fit(X, y)

VotingClassifier(estimators=[('lgb',
                              Pipeline(steps=[('preprocessor',
                                               ColumnTransformer(transformers=[('num',
                                                                                SimpleImputer(strategy='median'),
                                                                                ['age_approx',
                                                                                 'clin_size_long_diam_mm',
                                                                                 'tbp_lv_A',
                                                                                 'tbp_lv_Aext',
                                                                                 'tbp_lv_B',
                                                                                 'tbp_lv_Bext',
                                                                                 'tbp_lv_C',
                                                                                 'tbp_lv_Cext',
                                                                                 'tbp_lv_H',
                                                                                 'tbp_lv_Hext',
                                                                                 'tbp_lv_L',
                                                                                 'tbp_lv_Lext',
                                                                                 'tbp_lv_areaMM2',
                                                                                 'tbp_lv_area_perim_ratio',
                                                                                 'tb...
                                                             grow_policy=None,
                                                             importance_type=None,
                                                             interaction_constraints=None,
                                                             lambda=8.879624125465703,
                                                             learning_rate=0.08501257473292347,
                                                             max_bin=None,
                                                             max_cat_threshold=None,
                                                             max_cat_to_onehot=None,
                                                             max_delta_step=None,
                                                             max_depth=6,
                                                             max_leaves=None,
                                                             min_child_weight=None,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=None, ...))]))],
                 voting='soft')

### **Confusion Matrix**

In [28]:
# get classification report on the test set 
X_test = df_test_final[feature_cols]
y_test = df_test_final[target_col]
groups_test = df_test_final[group_col]

In [29]:
y_pred = estimator.fit(X, y).predict(X_test)
y_pred_proba = estimator.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba)}")
print(f"Partial AUC: {custom_metric(estimator, X_test, y_test)}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     60100
           1       0.31      0.14      0.19        59

    accuracy                           1.00     60159
   macro avg       0.65      0.57      0.59     60159
weighted avg       1.00      1.00      1.00     60159

ROC AUC: 0.9667816351278942
Partial AUC: 0.1739929496037677


## **Pre-trained EfficientNetB0 Model**

In [30]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import glob

In [31]:
# Optional: check if Metal GPU is available
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 1


In [32]:
# --- Parameters ---
IMG_SIZE = (224, 224)  # EfficientNetB0 default input size
SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
DATA_DIR = '/Users/lassestrandbygaard/msc-data-science/Exam Project/Solution Dataset/train'
TEST_DIR = '/Users/lassestrandbygaard/msc-data-science/Exam Project/Solution Dataset/test'
VAL_DIR = '/Users/lassestrandbygaard/msc-data-science/Exam Project/Solution Dataset/val'  

In [33]:
# Step 1: Define ImageDataGenerator (no validation split)
datagen = ImageDataGenerator(rescale=1./255)

# Step 2: Create training generator, test generator, and validation generator
train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

test_generator = datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)
val_generator = datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 284900 images belonging to 2 classes.
Found 60157 images belonging to 2 classes.
Found 60157 images belonging to 2 classes.


In [34]:
# Step 5: Get class labels
labels_index = train_generator.class_indices
print("Class indices:", labels_index)
print("Training samples:", train_generator.samples)

Class indices: {'benign': 0, 'malignant': 1}
Training samples: 284900


In [35]:
image_batch, label_batch = next(train_generator)
print("Image batch shape:", image_batch.shape)

Image batch shape: (32, 224, 224, 3)


In [36]:
NUM_CLASSES = len(labels_index)
NUM_CLASSES

2

In [37]:
# Define model-building function for binary classification
def build_model(freeze_base=True):
    inputs = layers.Input(shape=(SIZE, SIZE, 3))

    # Load EfficientNetB0 with ImageNet weights and set freeze_base to True or False based on the parameter
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
    base_model.trainable = not freeze_base

    # Add custom classification head
    x = layers.GlobalMaxPooling2D()(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3, name="top_dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="pred")(x)

    # Compile the model
    model = tf.keras.Model(inputs, outputs, name="EfficientNetB0_Binary")
    return model

In [38]:
# Build and compile model (Phase 1 - base frozen)
model = build_model(freeze_base=True)
optimizer = tf.keras.optimizers.legacy.Nadam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


2025-05-16 02:19:19.532538: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-05-16 02:19:19.532577: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-05-16 02:19:19.532593: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-05-16 02:19:19.532638: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-05-16 02:19:19.532671: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "EfficientNetB0_Binary"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 rescaling (Rescaling)       (None, 224, 224, 3)          0         ['input_1[0][0]']             
                                                                                                  
 normalization (Normalizati  (None, 224, 224, 3)          7         ['rescaling[0][0]']           
 on)                                                                                              
                                                                                                  
 rescaling_1 (Rescaling)     (None, 224, 224, 3)          0         ['normaliz

In [ ]:
# plot training history
def plot_history(history):
    plt.figure(figsize=(12, 4))
    
    # Plot training & validation accuracy values
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(loc='upper left')

    # Plot training & validation loss values
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

In [ ]:
# Train top layers only
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    verbose=1
)

In [ ]:
plot_history(model.history)

In [ ]:
# Phase 2: Unfreeze base and fine-tune
model = build_model(freeze_base=False)  # reuse architecture, but now with base_model.trainable = True
optimizer = tf.keras.optimizers.legacy.Nadam(learning_rate=1e-5)
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
# Continue training the full model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,  # Remaining epochs
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    verbose=1)

In [ ]:
plot_history(model.history)

In [ ]:
# Step 1: Predict on test set
y_pred_probs = model.predict(test_generator, steps=test_generator.samples // BATCH_SIZE + 1, verbose=1)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

In [ ]:
# Step 2: Get true labels
y_true = test_generator.classes  # returns label indices: 0 or 1

In [ ]:
# Step 3: Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)
labels = list(test_generator.class_indices.keys())

In [ ]:
# Step 4: Plot confusion matrix
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(cmap='Blues')
plt.title("Confusion Matrix - Test Set")
plt.show()

In [ ]:

# Step 5: Print classification report
print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
# Custom partial AUC function for EfficientNetB0 predictions
def compute_partial_auc(model, test_generator, min_tpr=0.80):
    # Step 1: Get predictions and true labels
    y_true = test_generator.classes
    y_pred_probs = model.predict(test_generator, steps=test_generator.samples // test_generator.batch_size + 1, verbose=1)

    # Step 2: Extract positive class probabilities if categorical
    if y_pred_probs.shape[1] == 2:
        y_pred_probs = y_pred_probs[:, 1]  # use only the malignant class prob

    # Step 3: Invert labels for partial AUC calculation (0 becomes 1, 1 becomes 0)
    v_gt = abs(y_true - 1)
    v_pred = np.array([1.0 - p for p in y_pred_probs])

    # Step 4: Define max FPR from min TPR
    max_fpr = abs(1 - min_tpr)

    # Step 5: Compute partial AUC and scale it
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)

    return partial_auc

In [ ]:
partial_auc = compute_partial_auc(model, test_generator)

In [ ]:
print(f"Partial AUC: {partial_auc:.4f}")